In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np, matplotlib.pyplot as plt
from config import TOPICS, DATA, RESULTS
import analyze as A

acts, topic, label, idx = A.load(DATA / "acts.npz")
topics = list(TOPICS)
acts.shape

## Layer sweep

In [ ]:
aucs = A.layer_sweep(acts, topic, label, idx, topics)
LAYER = int(aucs.mean(1).argmax())

plt.plot(aucs.mean(1), 'k')
plt.plot(aucs, alpha=.25)
plt.axvline(LAYER, ls='--', c='r'); plt.axhline(.5, ls=':', c='gray')
plt.xlabel('layer'); plt.ylabel('held-out AUC'); plt.show()

print(LAYER, aucs[LAYER].round(2))

**Gate:** drop any topic with AUC < 0.75. A default measured on a non-discriminating direction is noise.

In [ ]:
keep = [t for t, a in zip(topics, aucs[LAYER]) if a >= .75]
print(f"kept {len(keep)}/{len(topics)}:", keep)

## Where does the default sit?

In [ ]:
res = {t: A.calibrate(acts, topic, label, idx, t, LAYER) for t in keep}
means = np.array([res[t][0].mean() for t in keep])
cis = np.array([A.bootstrap_ci(res[t][0]) for t in keep])
o = np.argsort(means)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(range(len(keep)), means[o], color='#4a6fa5',
        xerr=np.abs(cis[o].T - means[o]), error_kw=dict(lw=1, ecolor='k'))
ax.set_yticks(range(len(keep))); ax.set_yticklabels([keep[i] for i in o])
ax.axvline(0, c='k', lw=1); ax.axvline(1, c='k', lw=1)
ax.set_xlabel('assumed expertise  (0 = novice anchor, 1 = expert anchor)')
plt.tight_layout(); plt.savefig(RESULTS / 'default_by_topic.png', dpi=160); plt.show()

## Is between-topic spread real?

In [ ]:
vals = [res[t][0] for t in keep]
between = np.var([v.mean() for v in vals])
within = np.mean([v.var() for v in vals])
print(f"between={between:.4f}  within={within:.4f}  ratio={between/within:.2f}")

# permutation null: shuffle topic assignment of neutral prompts
rng = np.random.default_rng(0)
flat = np.concatenate(vals); sizes = [len(v) for v in vals]
null = []
for _ in range(2000):
    p = rng.permutation(flat); s = np.split(p, np.cumsum(sizes)[:-1])
    null.append(np.var([x.mean() for x in s]))
print(f"p = {(np.array(null) >= between).mean():.4f}")

## Look at the data

In [ ]:
import json
rows = [json.loads(l) for l in open(DATA / 'prompts.jsonl')]
t = keep[0]
s = res[t][0]
nrows = [r for r in rows if r['topic'] == t and r['label'] == 'neutral']
for i in np.argsort(s)[[0, 1, -2, -1]]:
    print(f"{s[i]:+.2f}  {nrows[i]['text'][:120]}")